# 中山醫學大學附設醫院 SmartCoder 公開展示

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dechnology/csh-smartcoder-demo/blob/main/csh_smartcoder_api_demo.ipynb)

選「執行階段」→「全部執行」即可完成展示，不需輸入 API key 或設定環境變數。

> 請只使用下方內建的合成案例。請勿輸入真實病歷、姓名、身分證號、病歷號或其他個人資料。

公開展示用 `X-API-Key` 已內建在 Notebook，直接呼叫中山醫 SmartCoder 正式服務。展示憑證可由維運端隨時更換。

In [ ]:
from uuid import uuid4
import json

import requests

BASE_URL = "https://fhircsh.itri-nlp.tw/code_api/smartcoder"
PUBLIC_DEMO_X_API_KEY = "csh-public-demo-v1"  # 可公開、可更換的展示 token
COLAB_ORIGIN = "https://colab.research.google.com"
CODE_URL = f"{BASE_URL}/api/v1/snomed/coding"
RESULT_URL = f"{BASE_URL}/api/v1/snomed/results/{{request_id}}"
HEADERS = {
    "Content-Type": "application/json",
    "X-API-Key": PUBLIC_DEMO_X_API_KEY,
    "Origin": COLAB_ORIGIN,
}

print("中山醫 SmartCoder 正式服務已就緒：", BASE_URL)

In [ ]:
def assert_colab_cors(response):
    actual = response.headers.get("Access-Control-Allow-Origin")
    assert actual == COLAB_ORIGIN, f"CORS 不符：{actual!r}"


def safe_result(payload):
    return {
        "request_id": payload.get("request_id"),
        "polished_clinical_note": payload.get("polished_clinical_note"),
        "snomed_codings": [
            {k: v for k, v in item.items() if k != "source"}
            for item in payload.get("snomed_codings", [])
        ],
    }


request_id = str(uuid4())
demo_case = {
    "request_id": request_id,
    "encounter_type": "outpatient",
    "raw_clinical_note": (
        "58-year-old male presents with headache and dizziness. "
        "Home blood pressure is around 140-150/90. History includes "
        "type 2 diabetes mellitus, hypertension, and chronic kidney "
        "disease stage 3. Diabetes control is suboptimal; blood "
        "pressure remains borderline high. Plan: adjust Losartan, "
        "arrange UACR and fundus examination, and follow up in 4 weeks."
    ),
    "snomedct": [
        {
            "concept_id": "44054006",
            "term": "Diabetes mellitus type 2",
            "category": "disorder",
        },
        {
            "concept_id": "38341003",
            "term": "Hypertensive disorder",
            "category": "disorder",
        },
    ],
    "icd_codes": [
        {
            "code": "E11.9",
            "description": "Type 2 diabetes mellitus without complications",
        }
    ],
}

post_response = requests.post(
    CODE_URL, headers=HEADERS, json=demo_case, timeout=180
)
print("POST HTTP", post_response.status_code)
post_response.raise_for_status()
assert_colab_cors(post_response)
post_payload = post_response.json()
assert post_payload.get("request_id") == request_id, "POST request_id 不一致"
assert post_payload.get("snomed_codings"), "POST 編碼結果不得為空"

get_response = requests.get(
    RESULT_URL.format(request_id=request_id), headers=HEADERS, timeout=60
)
print("GET HTTP", get_response.status_code)
get_response.raise_for_status()
assert_colab_cors(get_response)
get_payload = get_response.json()
assert get_payload.get("request_id") == request_id, "GET request_id 不一致"

print(json.dumps(safe_result(post_payload), ensure_ascii=False, indent=2))
print("\n驗收通過：POST、同 request_id GET、編碼結果與 Colab CORS 皆正常。")